# ANAC LLM coding pipeline
# need to make sure you change the input and output file before you run
# API key stored in .env variable

In [ ]:
import os
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel, Field
from typing import Literal, Optional

pd.set_option("display.max_colwidth", 120)
tqdm.pandas()

# Load API key from .env
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY was not found in the environment. Add it to your .env file.")

print(f"API key loaded: {api_key[:10]}...")
client = OpenAI()


#Input variables/change PRN
INPUT_FILE = "ANAC_mp_loc.csv"   # change this
OUTPUT_FILE = "ANAC_output_llm.csv"  # change this
TEXT_COL = "Text"  
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")


TASKS_TO_RUN = [
    "severity",
    "cause",
    "heuristics",
    "participants_gender",
    "experience",
    "people_involved",
    "rescue_type",
]


In [ ]:

# Helper functions
def clean_text(value):
    ''' takes text from the df and returns clean text in string format'''
    if pd.isna(value):
        return ""
    return str(value).strip()


def apply_classifier(df, text_col, classify_fn, output_map, desc):
    '''Run though the whole dataframe by row, use caching for duplicate texts.'''
    cache = {}

    for out_col in output_map.values():
        if out_col not in df.columns:
            df[out_col] = None

    for idx, raw_text in tqdm(df[text_col].items(), total=len(df), desc=desc):
        text = clean_text(raw_text)
        if text in cache:
            result = cache[text]
        else:
            result = classify_fn(text)
            cache[text] = result

        if hasattr(result, "model_dump"):
            result = result.model_dump()
        elif not isinstance(result, dict):
            raise TypeError(f"Classifier returned unsupported type: {type(result)}")

        for src_field, df_col in output_map.items():
            df.at[idx, df_col] = result.get(src_field)

    return df


def heuristics_to_string(value):
    ''' convert a list into a cs string'''
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return "none"
    if isinstance(value, list):
        return "none" if len(value) == 0 else ",".join(value)
    text = str(value).strip()
    return "none" if text in {"", "[]", "none"} else text


def heuristics_count(value):
    ''' counts the number of heuristics applied'''
    if isinstance(value, list):
        return len(value)
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return 0
    text = str(value).strip()
    if text in {"", "none", "[]"}:
        return 0
    return len([h.strip() for h in text.split(",") if h.strip()])


In [ ]:

# output models

class SeverityLabel(BaseModel):
    severity: Literal["fatal", "serious", "moderate", "minor"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str


class CauseLabel(BaseModel):
    cause: Literal[
        "fall",
        "falling_object",
        "illness",
        "avalanche",
        "rappel_error",
        "weather_exposure",
        "equipment_failure",
        "stranded_lost",
        "unknown",
    ]
    confidence: int = Field(ge=0, le=100)
    reason: str


class HeuristicsLabel(BaseModel):
    heuristics: list[Literal[
        "familiarity",
        "acceptance",
        "consistency",
        "expert_halo",
        "scarcity",
        "social_facilitation",
        "back_to_the_barn",
        "blue_sky",
    ]]
    confidence: int = Field(ge=0, le=100)
    reason: str


class GenderLabel(BaseModel):
    participants_gender: Literal["men", "women", "both", "unknown"]
    confidence: int = Field(ge=0, le=100)
    reason: str


class ExperienceLabel(BaseModel):
    label: Literal["novice", "intermediate", "expert", "unknown"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str


class PeopleInvolved(BaseModel):
    total_people: Optional[int] = Field(default=None, ge=0)
    status: Literal["exact", "estimated", "unknown"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str


class RescueType(BaseModel):
    rescue_type: Literal["self_rescue", "professional_rescue", "unknown"]
    confidence: float = Field(ge=0.0, le=1.0)
    reason: str


In [ ]:

# Classifiers

def classify_severity(text: str):
    if not text:
        return SeverityLabel(severity="minor", confidence=0.0, reason="empty text")

    resp = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": """Classify the severity of a climbing accident.

Definitions:
- fatal: at least one person died
- serious: required immediate evacuation, rescue, or hospitalization
- moderate: injuries present but no urgent evacuation or hospitalization
- minor: no injuries or only trivial injuries

Rules:
- Always classify using the most severe outcome in the party
- fatal overrides all other categories
- evacuation by helicopter or ambulance implies serious
- being hospitalized implies serious
- mention of injury without evacuation implies moderate
- no injury or minor issues implies minor
- if unclear, choose the best-supported category""",
            },
            {"role": "user", "content": text},
        ],
        text_format=SeverityLabel,
    )
    return resp.output_parsed


def classify_cause(text: str):
    if not text:
        return CauseLabel(cause="unknown", confidence=0, reason="empty text")

    resp = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": """You are classifying the PRIMARY PROXIMAL CAUSE of a climbing accident.

Return only these fields: cause, confidence, reason.

Allowed cause labels:
- fall
- falling_object
- illness
- avalanche
- rappel_error
- weather_exposure
- equipment_failure
- stranded_lost
- unknown

Rules:
- Choose the single most immediate cause of the accident.
- Focus on what directly caused harm, not contributing factors.
- Do not invent new categories.
- If weather caused someone to fall, label it as fall.
- If gear failure caused a fall and the text emphasizes the system failure, use equipment_failure.
- If someone was both lost and later injured in a fall, choose the cause that directly triggered the injury.""",
            },
            {"role": "user", "content": text},
        ],
        text_format=CauseLabel,
    )
    return resp.output_parsed


def classify_heuristics(text: str):
    if not text:
        return HeuristicsLabel(heuristics=[], confidence=0, reason="empty text")

    resp = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": """You are identifying cognitive heuristics in climbing accident narratives.

Return only these fields: heuristics, confidence, reason.

Allowed heuristics:
- familiarity
- acceptance
- consistency
- expert_halo
- scarcity
- social_facilitation
- back_to_the_barn
- blue_sky

Rules:
- Be strict. Only include a heuristic when the text gives strong, direct evidence.
- If no heuristic is clearly supported, return an empty list.
- Prefer under-classification over over-classification.""",
            },
            {"role": "user", "content": text},
        ],
        text_format=HeuristicsLabel,
    )
    return resp.output_parsed


def classify_gender(text: str):
    if not text:
        return GenderLabel(participants_gender="unknown", confidence=0, reason="empty text")

    resp = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": """Classify the gender composition of the accident participants only.
Do not include rescuers, search and rescue personnel, medics, bystanders, or responders.

Allowed labels: men, women, both, unknown.

Rules:
- Use only explicit evidence in the text.
- Do not infer gender from names, pronouns alone if unclear, or stereotypes.
- If participant gender is not clearly stated, return unknown.
- Prefer under-classification over over-classification.""",
            },
            {"role": "user", "content": text},
        ],
        text_format=GenderLabel,
    )
    return resp.output_parsed


def classify_experience(text: str):
    if not text:
        return ExperienceLabel(label="unknown", confidence=0.0, reason="empty text")

    resp = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": """Classify climber experience level from an accident narrative.

Return:
- label: novice, intermediate, expert, or unknown
- confidence: 0 to 1
- reason: short explanation based only on text

Rules:
- Use only explicit or strongly implied evidence
- If unclear, return unknown""",
            },
            {"role": "user", "content": text},
        ],
        text_format=ExperienceLabel,
    )
    return resp.output_parsed


def classify_people_involved(text: str):
    if not text:
        return PeopleInvolved(total_people=None, status="unknown", confidence=0.0, reason="empty text")

    resp = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": """Count the total number of people directly involved in the accident event.
Count climbers, skiers, party members, victims, guides, and partners.
Do not count rescuers, wardens, pilots, or medical staff unless they were part of the accident party.
If the exact number is explicit, set status to exact.
If the number can be inferred but is not explicitly stated, set status to estimated.
If it cannot be determined, set total_people to null and status to unknown.""",
            },
            {"role": "user", "content": text},
        ],
        text_format=PeopleInvolved,
    )
    return resp.output_parsed


def classify_rescue_type(text: str):
    if not text:
        return RescueType(rescue_type="unknown", confidence=0.0, reason="empty text")

    resp = client.responses.parse(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": """Classify how the accident party was extricated from danger.

Use one label:
- self_rescue: the party, partners, or companions got themselves out without professional rescue extrication
- professional_rescue: organized rescuers, rangers, SAR, helicopter rescue, or medical rescue extricated them
- unknown: unclear from the text

Base the label on who performed the actual rescue/extrication.""",
            },
            {"role": "user", "content": text},
        ],
        text_format=RescueType,
    )
    return resp.output_parsed


In [ ]:

# Load data
df = pd.read_csv(INPUT_FILE).copy()
print(df.shape)
print(df.columns.tolist())

if TEXT_COL not in df.columns:
    raise KeyError(f"Column '{TEXT_COL}' was not found in {INPUT_FILE}")


In [ ]:

# Run selected tasks
if "severity" in TASKS_TO_RUN:
    df = apply_classifier(
        df,
        TEXT_COL,
        classify_severity,
        {
            "severity": "llm_severity",
            "confidence": "llm_severity_confidence",
            "reason": "llm_severity_reason",
        },
        desc="Coding severity",
    )
    print(df["llm_severity"].value_counts(dropna=False))

if "cause" in TASKS_TO_RUN:
    df = apply_classifier(
        df,
        TEXT_COL,
        classify_cause,
        {
            "cause": "llm_cause",
            "confidence": "llm_cause_confidence",
            "reason": "llm_cause_reason",
        },
        desc="Coding cause",
    )
    print(df["llm_cause"].value_counts(dropna=False))

if "heuristics" in TASKS_TO_RUN:
    df = apply_classifier(
        df,
        TEXT_COL,
        classify_heuristics,
        {
            "heuristics": "llm_heuristics",
            "confidence": "llm_heuristics_confidence",
            "reason": "llm_heuristics_reason",
        },
        desc="Coding heuristics",
    )
    df["llm_heuristics"] = df["llm_heuristics"].apply(heuristics_to_string)
    df["llm_heuristics_count"] = df["llm_heuristics"].apply(heuristics_count)
    print(df["llm_heuristics"].value_counts(dropna=False))

if "participants_gender" in TASKS_TO_RUN:
    df = apply_classifier(
        df,
        TEXT_COL,
        classify_gender,
        {
            "participants_gender": "llm_participants_gender",
            "confidence": "llm_participants_gender_confidence",
            "reason": "llm_participants_gender_reason",
        },
        desc="Coding participant gender",
    )
    print(df["llm_participants_gender"].value_counts(dropna=False))

if "experience" in TASKS_TO_RUN:
    df = apply_classifier(
        df,
        TEXT_COL,
        classify_experience,
        {
            "label": "llm_experience_level",
            "confidence": "llm_experience_confidence",
            "reason": "llm_experience_reason",
        },
        desc="Coding experience",
    )
    print(df["llm_experience_level"].value_counts(dropna=False))

if "people_involved" in TASKS_TO_RUN:
    df = apply_classifier(
        df,
        TEXT_COL,
        classify_people_involved,
        {
            "total_people": "llm_people_involved",
            "status": "llm_people_status",
            "confidence": "llm_people_confidence",
            "reason": "llm_people_reason",
        },
        desc="Counting people",
    )
    print(df["llm_people_status"].value_counts(dropna=False))

if "rescue_type" in TASKS_TO_RUN:
    df = apply_classifier(
        df,
        TEXT_COL,
        classify_rescue_type,
        {
            "rescue_type": "llm_rescue_type",
            "confidence": "llm_rescue_confidence",
            "reason": "llm_rescue_reason",
        },
        desc="Coding rescue type",
    )
    print(df["llm_rescue_type"].value_counts(dropna=False))

print(df.head())


In [ ]:
print (df.head())
# Save output


df.to_csv(OUTPUT_FILE, index=False)
print(f"Saved {OUTPUT_FILE}")
